# Anthropic provider

Goal: construct a keyless Anthropic loopback agent, show that HTTP plus a key fails closed, then optionally run a live HTTPS call.

Trust: T1 native provider plus T2 Python ports. Network: construct-only unless the next cell or `ANTHROPIC_API_KEY` has a key.

`api_key` stays positional. HTTPS is required when it is set. The binding does not read environment variables. Paste a key below, or keep using the environment.

Set `ANTHROPIC_MODEL` to a model id your key can reach; `GET https://api.anthropic.com/v1/models` lists them. Requests ask for at most 64,000 output tokens, the current Claude ceiling.


In [ ]:
ANTHROPIC_API_KEY = ""  # paste a key to run the live cell; never commit it
ANTHROPIC_MODEL = "claude-haiku-4-5-20251001"

In [ ]:
import finstack_ai

agent = await finstack_ai.Agent.anthropic(
    "http://127.0.0.1:9",
    "fixture-model",
    instruction="Answer concisely.",
)
print(agent.compact_capability_catalog())

In [ ]:
try:
    await finstack_ai.Agent.anthropic(
        "http://127.0.0.1:9",
        "fixture-model",
        api_key="notebook-http-key",
    )
except finstack_ai.ConfigurationError as error:
    assert "notebook-http-key" not in str(error)
    assert "notebook-http-key" not in repr(error)
    print(error.code)

In [ ]:
from pydantic import BaseModel

from _support import live_value

import finstack_ai


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.anthropic",
    name="math",
)

api_key = live_value(ANTHROPIC_API_KEY, "ANTHROPIC_API_KEY")
if api_key:
    live_agent = await finstack_ai.Agent.anthropic(
        "https://api.anthropic.com",
        ANTHROPIC_MODEL,
        api_key,
        "Answer with the tool result only.",
        toolsets=[tools],
    )
    result = await live_agent.run("Add 20 and 22")
    print(result.text)
else:
    print("skipped: ANTHROPIC_API_KEY unset")